# Covering paint to low mass, with the generated closure-radius aperture

Extends the covering paint ([`covering_paint_pk.ipynb`](covering_paint_pk.ipynb))
down to **all halos $\geq$ a chosen floor** (default $10^{10}$), and replaces the
fixed $4\,R_{200c}$ paste aperture with the **closure radius** $R_c$ — the radius
where each halo's *generated* cumulative baryon fraction $f_b(<r)$ returns to
cosmic $\Omega_b/\Omega_m$. Because $R_c$ is read off BIND's **own generated
maps**, it needs no truth and is fully deployable.

Why: painting every halo to $10^{10}$ is cheap (covering ⇒ generations set by
tiling, not halo count) but **overshoots** the total-matter $P(k)$ with a fixed
$4R_{200}$ aperture. The generated closure radius is mass-dependent (larger for
groups, smaller for dwarfs) and reduces the overshoot at every scale.

## 0. Config + load DMO/truth maps + DM FoF catalog

In [ ]:
import h5py, numpy as np, time
from pathlib import Path
import matplotlib.pyplot as plt
from bind.inference.artifacts import load_full_maps, load_halo_catalog
from bind.inference.pipeline import extract_multiscale, circular_taper_weight
from bind.inference.paint import Model
from bind.metrics import power_spectrum_pylians_2d
from scipy.ndimage import gaussian_filter
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

SIM_L, NPIX, PP = 50.0, 1024, 128
MPP = SIM_L / NPIX; PPM = NPIX / SIM_L; HALF = (PP / 2) * MPP
R200_FAC, TAPER = 4.0, 0.15
F_COSMIC = 0.049 / 0.30                       # Omega_b/Omega_m (CV fiducial)
# inter-halo gas background: fill the space between halo apertures with diffuse
# gas tracing the DMO web at the cosmic gas fraction, so gas is connected (no hard
# taper edges). Void DM is split (1-F_GAS_BG)*DMO so TOTAL matter is conserved ->
# total-matter P(k) unchanged; only the gas channel changes.
F_GAS_BG, GAS_BG_SIGMA = 0.15, 2.0
FLOOR = 1e10                                  # mass floor for the main paint
CACHE = Path("covering_lowmass_cache"); CACHE.mkdir(exist_ok=True)

sd = Path("/mnt/home/mlee1/ceph/fm_lowmass/CV/sim_0/snap_090")
dmo, truth = load_full_maps(sd / "full_maps.npz")
GAS_BG = F_GAS_BG * gaussian_filter(dmo, GAS_BG_SIGMA, mode="wrap").astype(np.float32)
_h, _, _, _ = load_halo_catalog(sd / "mass_threshold_1p000e12/halo_catalog.npz")
sim_params = np.asarray(_h[0]["params"], np.float32)   # p14=0 already applied
FOF = "/mnt/ceph/users/camels/FOF_Subfind/IllustrisTNG_DM/L50n512/CV/CV_0/fof_subhalo_tab_090.hdf5"
with h5py.File(FOF, "r") as h:
    M200_ALL = h["Group/Group_M_Crit200"][:] * 1e10
    POS_ALL = h["Group/GroupPos"][:] / 1e3            # Mpc/h
    R200_ALL = h["Group/Group_R_Crit200"][:] / 1e3    # Mpc/h
print(f"FoF: {(M200_ALL>FLOOR).sum()} halos > {FLOOR:.0e}; truth {truth.shape}")

model = Model.from_files(
    "/mnt/home/mlee1/ceph/fm_runs/fm_thermo/checkpoints/kept/keep_epoch064_ema.ckpt",
    "/mnt/home/mlee1/ceph/fm_runs/fm_thermo/norm_stats.npz", device="cuda")

## 1. Greedy geometric covering plan (steps 2–6)

Largest-first; a halo joins a box iff its full $4R_{200}$ aperture fits. Returns
per-halo `(box, pixel position)` and the center halo index of each generation.

In [ ]:
def plan_covering(floor):
    sel = np.where(M200_ALL > floor)[0]
    C = POS_ALL[sel, :2].astype(float); R = R200_ALL[sel]; ap = R200_FAC * R
    order = sel[np.argsort(-M200_ALL[sel])]
    order_local = np.argsort(-M200_ALL[sel])
    cov = np.zeros(len(sel), bool); ci = []; box = np.full(len(sel), -1)
    for oi in order_local:
        if cov[oi]:
            continue
        ci.append(oi); unc = np.where(~cov)[0]
        d = C[unc] - C[oi]; d -= SIM_L * np.round(d / SIM_L)
        fits = (np.abs(d[:, 0]) + ap[unc] < HALF) & (np.abs(d[:, 1]) + ap[unc] < HALF)
        fits[unc == oi] = True
        box[unc[fits]] = len(ci) - 1; cov[unc[fits]] = True
    px = (C[:, 0] * PPM).astype(int) % NPIX; py = (C[:, 1] * PPM).astype(int) % NPIX
    return dict(sel=sel, M=M200_ALL[sel], R=R, ci=np.array(ci), box=box, px=px, py=py)

P = plan_covering(FLOOR)
print(f"floor {FLOOR:.0e}: {len(P['sel'])} halos -> {len(P['ci'])} generations "
      f"({len(P['sel'])/len(P['ci']):.0f}x)")

## 2. Generate at the covering centers (cached)

In [ ]:
def generate_centers(P, floor):
    f = CACHE / f"gen_{floor:.0e}.npz"
    if f.exists():
        return np.load(f)["gen"]
    cut = []
    for i in P["ci"]:
        cond, ls = extract_multiscale(dmo, P["px"][i], P["py"][i], target_res=PP, mpc_per_pix=MPP)
        cut.append({"condition": cond, "large_scale": ls})
    g = model.generate(cut, sim_params, n_steps=20, batch_size=32,
                       use_amp=True, progress=False)[:, :3].astype(np.float32)
    np.savez_compressed(f, gen=g); return g

G = generate_centers(P, FLOOR)
print("generated", G.shape, "| global f_b =",
      round((G[:, 1].sum() + G[:, 2].sum()) / G.sum(), 4), "(cosmic", round(F_COSMIC, 4), ")")

## 3. The generated closure radius $R_c$ (the default aperture)

For each halo, roll its assigned generation so the halo is centred, then find the
smallest radius where cumulative $f_b(<r)=\Omega_b/\Omega_m$. $f_b$ is a ratio, so
$R_c$ is independent of the later mass-match. Capped so the aperture stays inside
the patch.

In [ ]:
_yy, _xx = np.mgrid[0:PP, 0:PP]
_RR = np.round(np.hypot(_xx - PP // 2, _yy - PP // 2)).astype(int).ravel(); _NR = _RR.max() + 1

def rolled_patch(P, G, j):
    g = P["box"][j]; c = P["ci"][g]
    dpx = (P["px"][j] - P["px"][c] + NPIX // 2) % NPIX - NPIX // 2
    dpy = (P["py"][j] - P["py"][c] + NPIX // 2) % NPIX - NPIX // 2
    cap = max(3, PP // 2 - 2 - max(abs(dpx), abs(dpy)))
    return np.roll(G[g], shift=(-dpx, -dpy), axis=(1, 2)), cap

def closure_radius_px(patch, cap):
    bar = (patch[1] + patch[2]).ravel(); tot = patch.sum(0).ravel()
    fb = np.cumsum(np.bincount(_RR, weights=bar, minlength=_NR)) / \
         np.maximum(np.cumsum(np.bincount(_RR, weights=tot, minlength=_NR)), 1e-30)
    hit = np.where(fb >= F_COSMIC)[0]
    return int(np.clip(hit[0] if len(hit) else _NR - 1, 2, cap))

def aperture_radii(P, G, mode):
    n = len(P["sel"]); apr = np.zeros(n)
    for j in range(n):
        if P["box"][j] < 0:
            continue
        if mode == "4R200":
            apr[j] = np.clip(R200_FAC * P["R"][j] * PPM, 1.5, PP // 2 - 2)
        else:  # generated closure radius
            patch, cap = rolled_patch(P, G, j)
            apr[j] = closure_radius_px(patch, cap)
    return apr

apr_rc = aperture_radii(P, G, "closure")
apr_4 = aperture_radii(P, G, "4R200")

# R_c(M) relation
lm = np.log10(P["M"]); eb = np.arange(10, 14.01, 0.3)
mids, rcM, r4M = [], [], []
for lo, hi in zip(eb[:-1], eb[1:]):
    m = (lm >= lo) & (lm < hi) & (P["box"] >= 0)
    if m.sum() < 5: continue
    mids.append(10 ** (0.5 * (lo + hi))); rcM.append(np.median(apr_rc[m]) * MPP); r4M.append(np.median(apr_4[m]) * MPP)
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.loglog(mids, rcM, "o-", label="$R_c$ (generated, $f_b$=cosmic)")
ax.loglog(mids, r4M, "s--", label="$4\\,R_{200}$")
ax.set(xlabel="$M_{200c}$", ylabel="aperture radius [Mpc/h]",
       title="Generated closure radius vs mass"); ax.legend(); plt.show()

## 4. Paint (custom low-memory) and compare $P(k)$ to hydro truth

In [ ]:
def paint(P, G, aper_px):
    canvas = np.zeros((3, NPIX, NPIX), np.float32); wacc = np.zeros((NPIX, NPIX), np.float32)
    wc = {}; ar = np.arange(PP)
    def gw(a):
        k = round(a * 2) / 2
        if k not in wc: wc[k] = circular_taper_weight(PP, r_pix=k, taper_frac=TAPER).astype(np.float32)
        return wc[k]
    for j in range(len(P["sel"])):
        if P["box"][j] < 0: continue
        patch, _ = rolled_patch(P, G, j); w = gw(aper_px[j])
        ix = (P["px"][j] - PP // 2 + ar) % NPIX; iy = (P["py"][j] - PP // 2 + ar) % NPIX
        dw = dmo[np.ix_(ix, iy)]
        patch = patch * (float((dw * w).sum()) / (float((patch.sum(0) * w).sum()) + 1e-30))
        gg = np.ix_(ix, iy)
        for ch in range(3): canvas[ch][gg] += patch[ch] * w
        wacc[gg] += w
    canvas /= np.where(wacc > 0, wacc, 1.0)[None]; alpha = np.clip(wacc, 0, 1)
    comp = np.empty((3, NPIX, NPIX), np.float32)
    # DM void = (1-f_gas)*DMO, Gas void = f_gas*smooth(DMO) -> total conserved,
    # gas connected across the web (removes hard inter-halo edges).
    comp[0] = (1 - alpha) * (1 - F_GAS_BG) * dmo + alpha * canvas[0]
    comp[1] = (1 - alpha) * GAS_BG + alpha * canvas[1]
    comp[2] = alpha * canvas[2]
    comp *= dmo.sum() / (comp.sum() + 1e-30)
    return comp, 100 * (alpha > 0.01).mean()

def pk(f):
    k, p, _ = power_spectrum_pylians_2d(f, box_size=SIM_L, MAS="None"); return k, p

comp_rc, cov_rc = paint(P, G, apr_rc)
comp_4, cov_4 = paint(P, G, apr_4)
k, pt = pk(truth.sum(0)); _, p4 = pk(comp_4.sum(0)); _, prc = pk(comp_rc.sum(0))
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].loglog(k, pt, "k-", label="hydro truth")
ax[0].loglog(k, p4, "C0-", label=f"4R200 (cov {cov_4:.0f}%)")
ax[0].loglog(k, prc, "C3-", label=f"$R_c$ generated (cov {cov_rc:.0f}%)")
ax[0].set(xlabel="k [h/Mpc]", ylabel="P(k)", title=f"Total-matter P(k), floor {FLOOR:.0e}"); ax[0].legend()
ax[1].semilogx(k, p4 / pt, "C0-", label="4R200 / truth")
ax[1].semilogx(k, prc / pt, "C3-", label="$R_c$ / truth")
ax[1].axhline(1, color="k", lw=0.8); ax[1].set(xlabel="k [h/Mpc]", ylabel="ratio", ylim=(0.8, 1.2)); ax[1].legend()
plt.tight_layout(); plt.show()
for nm, p in [("4R200", p4), ("R_c", prc)]:
    print(f"  {nm:6s}:", " ".join(f"{l} {np.median(p[(k>=lo)&(k<hi)]/pt[(k>=lo)&(k<hi)]):.3f}"
          for l, (lo, hi) in {"large": (0, 2), "mid": (2, 10), "small": (10, 40)}.items()))

### Showcase — full box: truth vs BIND2 vs residual

The $R_c$ composite (`comp_rc`) beside the hydro truth, per channel, over the
full 50 Mpc/h box, with the fractional residual $(\mathrm{BIND2}-\mathrm{truth})
/\mathrm{truth}$. Painting all halos $\geq10^{10}$ fills in the whole cosmic web
(vs the sparse bubbles a high mass floor leaves).

In [ ]:
def _lg(im):
    p = im[im > 0]; return np.log10(np.clip(im, p.min() if len(p) else 1e-30, None))
cols = ["DMO (input)", "DM_hydro", "Gas", "Stars"]
trow = [dmo, truth[0], truth[1], truth[2]]; brow = [dmo, comp_rc[0], comp_rc[1], comp_rc[2]]
fig, axes = plt.subplots(3, 4, figsize=(13, 9.8))
for a, t, im in zip(axes[0], cols, trow):
    v = _lg(im); a.imshow(v, cmap="magma", vmin=np.percentile(v, 5), vmax=np.percentile(v, 99.7))
    a.set_title(t); a.set_xticks([]); a.set_yticks([])
axes[0, 0].set_ylabel("Truth", fontsize=13)
for a, im, tr in zip(axes[1], brow, trow):
    ref = _lg(tr); a.imshow(_lg(im), cmap="magma", vmin=np.percentile(ref, 5), vmax=np.percentile(ref, 99.7))
    a.set_xticks([]); a.set_yticks([])
axes[1, 0].set_ylabel("BIND2", fontsize=13)
axes[2, 0].axis("off")
axes[2, 0].text(0.5, 0.5, "residual\n(BIND2 - truth)/truth", ha="center", va="center",
                transform=axes[2, 0].transAxes, fontsize=11)
for a, ch in zip(axes[2, 1:], [0, 1, 2]):
    thr = np.percentile(truth[ch][truth[ch] > 0], 20)
    fr = np.where(truth[ch] > thr, (comp_rc[ch] - truth[ch]) / np.where(truth[ch] > 0, truth[ch], np.nan), np.nan)
    im = a.imshow(fr, cmap="RdBu_r", vmin=-0.5, vmax=0.5); a.set_xticks([]); a.set_yticks([])
fig.colorbar(im, ax=axes[2, 1:].tolist(), fraction=0.025, pad=0.02).set_label("fractional residual")
fig.suptitle(f"BIND2 covering paint ($R_c$, all halos >{FLOOR:.0e}) — CV/sim_0 (50 Mpc/h)", y=0.995)
plt.show()

## 5. Floor sweep with the closure-radius aperture (b)

Does the closure radius flatten the optimum — i.e. is $10^{10}$ usable now? We
sweep the mass floor and compare mid-$k$ error for $4R_{200}$ vs $R_c$.

In [ ]:
floors = [1e12, 3e11, 1e11, 3e10, 1e10]
mid = (k >= 2) & (k < 10)
rows = []
for mm in floors:
    Pf = plan_covering(mm); Gf = generate_centers(Pf, mm)
    a4 = aperture_radii(Pf, Gf, "4R200"); arc = aperture_radii(Pf, Gf, "closure")
    _, q4 = pk(paint(Pf, Gf, a4)[0].sum(0)); _, qrc = pk(paint(Pf, Gf, arc)[0].sum(0))
    rows.append((mm, len(Pf["sel"]), len(Pf["ci"]),
                 np.median(q4[mid] / pt[mid]), np.median(qrc[mid] / pt[mid])))
    print(f"floor {mm:.0e}: {rows[-1][1]:5d} halos, {rows[-1][2]:3d} gen  "
          f"mid-k  4R200 {rows[-1][3]:.3f}  R_c {rows[-1][4]:.3f}")
fl = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogx(fl, [100 * (r[3] - 1) for r in rows], "s-", label="4R200")
ax.semilogx(fl, [100 * (r[4] - 1) for r in rows], "o-", label="$R_c$ generated")
ax.axhline(0, color="k", lw=0.8); ax.invert_xaxis()
ax.set(xlabel="mass floor [Msun/h]", ylabel="mid-k P(k) error vs truth [%]",
       title="Floor sweep: closure radius vs 4R200"); ax.legend(); plt.show()

## 6. Is BIND mis-painting dwarfs? (c)

Stack the **projected** cumulative $f_b(<r)$ from BIND's generated patches vs the
hydro truth across mass bins (same projection ⇒ same background systematic, so
the comparison is fair). This tests whether the small dwarf $R_c$ reflects a BIND
fidelity error or just projection.

In [ ]:
_r = np.round(np.hypot(_xx - PP // 2, _yy - PP // 2)).astype(int)
def stack_fb(idx, source):  # source: 'gen' or 'truth'
    cb = np.zeros(_NR); ct = np.zeros(_NR)
    for j in idx:
        if source == "gen":
            patch, _ = rolled_patch(P, G, j); bar = patch[1] + patch[2]; tot = patch.sum(0)
        else:
            ix = (P["px"][j] - PP // 2 + np.arange(PP)) % NPIX
            iy = (P["py"][j] - PP // 2 + np.arange(PP)) % NPIX
            t = truth[:, ix][:, :, iy]; bar = t[1] + t[2]; tot = t.sum(0)
        cb += np.bincount(_r.ravel(), weights=bar.ravel(), minlength=_NR)
        ct += np.bincount(_r.ravel(), weights=tot.ravel(), minlength=_NR)
    return np.cumsum(cb) / np.maximum(np.cumsum(ct), 1e-30)

rr_mpc = np.arange(_NR) * MPP
rng = np.random.default_rng(0)
bins = [("dwarf $10^{10}$–$10^{11}$", 1e10, 1e11), ("MW $10^{11}$–$10^{12}$", 1e11, 1e12),
        ("group $10^{12}$–$10^{13}$", 1e12, 1e13)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (nm, lo, hi) in zip(axes, bins):
    idx = np.where((P["M"] >= lo) & (P["M"] < hi) & (P["box"] >= 0))[0]
    if len(idx) > 2000: idx = rng.choice(idx, 2000, replace=False)
    ax.plot(rr_mpc, stack_fb(idx, "gen"), "C3-", label="BIND generated")
    ax.plot(rr_mpc, stack_fb(idx, "truth"), "k-", label="hydro truth")
    ax.axhline(F_COSMIC, color="grey", ls="--", label="cosmic")
    ax.set(xlabel="r [Mpc/h]", xlim=(0, 2), title=f"{nm} (n={len(idx)})")
    if lo == 1e10: ax.set_ylabel("cumulative $f_b(<r)$"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
# Finding: dwarfs/MW match truth almost exactly (small dwarf R_c is a projection/
# resolution artifact -- sub-pixel dwarfs are background-dominated). The only real
# gap is group cores, where BIND under-depletes (more central gas than truth).

## 7. Takeaway

- **Default aperture = generated closure radius $R_c$** (§3): read off BIND's own
  maps where $f_b(<r)=\Omega_b/\Omega_m$, no truth needed. Mass-dependent — larger
  for groups, smaller for dwarfs.
- **It reduces the low-mass overshoot** at every scale vs fixed $4R_{200}$ (§4), a
  controlled comparison (same generations) so the gain is real, not sampler noise.
- **Floor sweep (§5):** $R_c$ helps at every floor and most at $10^{10}$
  (mid-$k$ ~+8%→+5%), pulling the lowest floor back to competitive — so painting
  all halos to $10^{10}$ becomes viable at 132× fewer GPU calls.
- **Dwarf check (§6):** BIND's projected dwarf/MW $f_b(<r)$ **matches truth** —
  dwarfs are not mis-painted; the small dwarf $R_c$ is a projection/resolution
  artifact (sub-pixel, background-dominated). The only real fidelity gap is at
  **group cores**, where BIND slightly under-depletes (more central gas than
  truth) — the over-concentration that drives most of the residual overshoot and
  that the larger group $R_c$ aperture partly compensates.